# Демо проектных упражнений модуля 10 — Контекстный ключ кэша и ошибка выхода внимания

**Демо-реализация, не готовое решение полномасштабного проекта.** Демо 1 сравнивает страницы и точные совпадения токеновых префиксов; различает долю запросов с попаданием и долю повторно использованных токенов. Демо 2 измеряет ошибку внимания после учебного квантования и вытеснения. Это CPU-механизмы на синтетических данных, не реализации vLLM, KIVI, TurboQuant или H2O.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'

plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":12, "axes.titlesize":15, "axes.labelsize":13, "xtick.labelsize":11, "ytick.labelsize":11, "figure.dpi":150})


## Демо 1. Свободный хвост и совпавший префикс

Сначала выделим страницы по 16 слотов для 500 случайных длин и сравним их с резервированием по 4096. Затем каждый из 100 пользователей отправит пять запросов по 48 токенов. Первые 32 токена принадлежат одному из десяти общих шаблонов, а оставшиеся различают пользователя и ход. Ключ записи включает весь предшествующий префикс, область доступа и ревизию модели: одного совпадения локального блока недостаточно.

In [ ]:
import hashlib
rng = np.random.default_rng(SEED)
lengths = rng.integers(64, 4097, size=500)
page = 16
allocated = ((lengths + page - 1) // page) * page
reserved = np.full_like(lengths, 4096)
assert np.all(allocated - lengths < page)
print("waste / allocated:", (allocated-lengths).sum()/allocated.sum(),
      "fixed-reservation waste:", (reserved-lengths).sum()/reserved.sum())
assert (2048 + page - 1) // page * page == 2048

# Complete block boundaries. The key includes ALL preceding token IDs.
# No real KV tensors, tokenizer, network, TTL, eviction, or serving engine.
def prefix_key(tokens, end, scope, revision):
    return (scope, revision, tuple(tokens[:end]))

def run_prefix_trace(isolated):
    cache = set(); hit_requests = cached_tokens = input_tokens = 0
    for user in range(100):
        for turn in range(5):
            tokens = [100 + user % 10] + list(range(31)) + [1000+user, 2000+turn] + list(range(14))
            scope = user if isolated else "shared-public-template"
            hit = 0
            for end in range(page, len(tokens)+1, page):
                if prefix_key(tokens,end,scope,"model-r1") in cache:
                    hit = end
                else:
                    break
            for end in range(page,len(tokens)+1,page):
                cache.add(prefix_key(tokens,end,scope,"model-r1"))
            hit_requests += hit > 0
            cached_tokens += hit
            input_tokens += len(tokens)
    return hit_requests/500, cached_tokens/input_tokens

shared = run_prefix_trace(False); isolated = run_prefix_trace(True)
assert np.isclose(shared[0], .98) and np.isclose(isolated[0], .8)
assert np.isclose(shared[1], 490*32/(500*48))
print("request-hit fraction / reused-input fraction:", shared, isolated)
a = list(range(32)); b = a.copy(); b[0] = 999
assert a[16:32] == b[16:32]  # same block content, different preceding context
assert prefix_key(a,32,"scope","r1") != prefix_key(b,32,"scope","r1")
assert prefix_key(a,32,"scope","r1") != prefix_key(a,32,"scope","r2")
print("Context and model-revision identity: PASS")

Запросов с попаданием 98% при общем кэше и 80% при изоляции по пользователю. Но доли повторно использованных входных токенов ниже: около 65.33% и 53.33%, потому что последние 16 токенов каждого запроса новые. Ни один из этих показателей сам по себе не предсказывает ускорение. В первой части длина 2048 заполняет страницы точно; средняя потеря на случайных длинах относится уже к другому распределению.

## Демо 2. Ошибка внимания и смена запроса

Создадим 128 ключей и значений ширины 32 и двенадцать запросов. Учебный квантователь использует симметричные целые коды с отдельным масштабом на строку. После восстановления сравним выход внимания с исходным. Во втором опыте прошлый запрос выделяет позицию 0, а новый требует позицию 90. Бюджет в 16 позиций позволяет увидеть, что случится, если решение о сохранении принято по прошлому или только по месту токена.

In [ ]:
rng = np.random.default_rng(SEED + 10)
n, d = 128, 32
keys = rng.normal(size=(n,d)); values = rng.normal(size=(n,d))
queries = rng.normal(size=(12,d))
def attention(q,k,v):
    logits = q @ k.T / np.sqrt(k.shape[1])
    logits -= logits.max(axis=-1,keepdims=True)
    prob = np.exp(logits); prob /= prob.sum(axis=-1,keepdims=True)
    return prob @ v, prob

def fake_quantize_rows(x,bits):
    limit = 2**(bits-1)-1
    scale = np.max(np.abs(x),axis=-1,keepdims=True)/limit
    scale = np.where(scale == 0, 1.0, scale)
    codes = np.clip(np.rint(x/scale),-limit,limit).astype(np.int8)
    # Float64 dequantization here is a numerical experiment, not a packed kernel.
    return codes.astype(float)*scale

reference,_ = attention(queries,keys,values)
for bits in (8,3):
    kq,vq = fake_quantize_rows(keys,bits),fake_quantize_rows(values,bits)
    out,_ = attention(queries,kq,vq)
    error = np.linalg.norm(out-reference)/np.linalg.norm(reference)
    ideal_codes = 2*n*d*bits/8
    fp32_scales = 2*n*4
    print(f"{bits} bits: relative output error={error:.5f}; ideal packed codes+scales={ideal_codes+fp32_scales:.0f} B")
    if bits == 8: assert error < .03
np.testing.assert_array_equal(fake_quantize_rows(np.zeros((2,4)),3),0)

# A controlled query shift: past attention favors token 0; the next query needs token 90.
k = np.zeros((n,d)); k[0,0] = 12; k[90,1] = 12
v = np.zeros((n,2)); v[0,0] = 1; v[90,1] = 1
past = np.zeros((1,d)); past[0,0] = 4
future = np.zeros((1,d)); future[0,1] = 4
_, mass = attention(past,k,v)
budget = 16
ranked = np.argsort(-mass[0],kind="stable")[:budget]
streaming = np.r_[np.arange(4),np.arange(n-(budget-4),n)]
exact,_ = attention(future,k,v)
for name,idx in (("past top-mass",ranked),("sink + recent",streaming)):
    out,_ = attention(future,k[idx],v[idx])
    error = np.linalg.norm(out-exact)
    print(name,"retains future target:",90 in idx,"output error:",error)
    assert 90 not in idx and error > .9
# Keeping the target has a low error on this example, but requires future knowledge.
oracle = np.r_[90,np.arange(budget-1)]
recovered,_ = attention(future,k[oracle],v[oracle])
assert np.linalg.norm(recovered-exact) < .03
print("Query-shift failure and oracle control: PASS")

Выведенный размер считает идеальные упакованные коды и масштабы float32. Он не равен памяти NumPy-массивов: восстановленные значения имеют тип float64. При смене запроса обе политики теряют позицию 90 и дают большую ошибку выхода. Контроль с сохранённой позицией восстанавливает ответ, но использует знание будущего запроса. Он объясняет причину ошибки, а не предлагает реализуемую политику предсказания будущего.

## Переход к полному проекту

Для У4 сначала измените распределение длин и число шаблонов. Для У5 измените битность, бюджет и позицию будущей цели. Необязательный опыт в движке должен фиксировать поток поступления, прогрев, ревизии и SLO; измеряйте TTFT/ITL, goodput, память и качество ответов. Норма сохранённых K/V не заменяет оценку ответа.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.